# The Complete Training Flow (Code-based Deep Explanation)

අපි මේ වෙනකන් Loss, Cost, Activations, Optimizers සහ Backpropagation ගැන වෙන වෙනම න්‍යායාත්මකව (Theory) ඉගෙන ගත්තා. හැබැයි අපි Keras වල `model.fit()` කියලා කමාන්ඩ් එක ගැහුවාම මේ ඔක්කොම දේවල් තිරයෙන් පිටුපස (under the hood) ස්වයංක්‍රීයව සිද්ධ වෙනවා.

අද අපි `model.fit()` පාවිච්චි කරන්නේ නැතුව, මේ සම්පූර්ණ ක්‍රියාවලියම (Forward Pass ඉඳන් Backward Pass වෙනකන්) අපේම අතින් Code කරලා බලමු (Custom Training Loop). එතකොට Neural Network එකක් ඇත්තටම ඉගෙනගන්න විදිහ ඔයාට 100% ක් පැහැදිලි වෙයි.

In [1]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

# Mac වල Kernel Crash වෙන එක නවත්වන්න CPU එක විතරක් පාවිච්චි කිරීම
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
try:
    tf.config.set_visible_devices([], 'GPU')
except:
    pass

---
## 1. සරල දත්ත ගොනුවක් සෑදීම (Dummy Dataset)

අපි සරල Classification ප්‍රශ්නයක් ගමු. X අක්ෂයේ අගයයි, Y අක්ෂයේ අගයයි එකතු කළාම උත්තරේ 0 ට වඩා වැඩියි නම් Class 1 (නිල් පාට), නැත්නම් Class 0 (රතු පාට) කියලා හිතමු.

In [2]:
# Training Data 1000ක් හදමු
X_train = np.random.randn(1000, 2).astype(np.float32)
# X1 + X2 > 0 නම් 1, නැත්නම් 0
y_train = (X_train[:, 0] + X_train[:, 1] > 0).astype(np.float32).reshape(-1, 1)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

X_train shape: (1000, 2)
y_train shape: (1000, 1)


---
## 2. Model එක, Loss Function එක සහ Optimizer එක හැඳින්වීම

අපි layers දෙකක් තියෙන සරල Neural Network එකක් හදමු.
- **Activation Function:** Hidden layer එකට `ReLU` (සෘණ අගයන් අයින් කරන්න). අවසාන Layer එකට `Sigmoid` (උත්තරේ 0 ත් 1 ත් අතර සම්භාවිතාවක් විදිහට ගන්න).
- **Loss Function:** Classes 2ක් (0 සහ 1) තියෙන නිසා අපි `BinaryCrossentropy` පාවිච්චි කරනවා.
- **Optimizer:** අපි අර කලින් ඉගෙන ගත්ත ජනප්‍රියම `Adam` optimizer එක ගන්නවා.

In [3]:
# 1. Model එක සෑදීම
model = tf.keras.Sequential([
    tf.keras.layers.Dense(4, activation='relu', input_shape=(2,)), # Hidden Layer (Neurons 4)
    tf.keras.layers.Dense(1, activation='sigmoid')                 # Output Layer
])

# 2. Loss (Cost) Function එක නිර්වචනය කිරීම
loss_fn = tf.keras.losses.BinaryCrossentropy()

# 3. Optimizer එක නිර්වචනය කිරීම
optimizer = tf.keras.optimizers.Adam(learning_rate=0.01)

/Users/adithyabandara/miniconda3/envs/stt/lib/python3.11/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


### Layers දාන විදිහ සහ ඒවා තීරණය කරන්නේ කොහොමද? (Network Architecture Design)

ගොඩක් අයට තියෙන ලොකුම ප්‍රශ්නයක් තමයි "මම Hidden Layers කීයක් දාන්න ඕනෙද? ඒවගේ Neurons කීයක් දාන්න ඕනෙද?" කියන එක. මේකට නිශ්චිත ගණිතමය සූත්‍රයක් නෑ (මේක කලාවක් වගේ දෙයක්). හැබැයි අපි පාවිච්චි කරන සම්මත නීති (Rules of thumb) ටිකක් තියෙනවා:

1. **Input Layer එක (ආදානය):**
   - මේක තීරණය කරන්නේ ඔයා. අපේ දත්ත වල Columns (Features) කීයක් තියෙනවද කියන එක මත. උදාහරණයක් විදිහට අපේ X වල තියෙන්නේ ඉලක්කම් 2යි නම්, `input_shape=(2,)` වෙනවා.

2. **Output Layer එක (ප්‍රතිදානය - අනිවාර්ය නීති):**
   - මේක තීරණය වෙන්නේ ඔයා විසඳන්න හදන ප්‍රශ්නය මතයි.
   - **Binary Classification (ඔව්/නෑ, 0/1):** Neuron 1 යි. Activation එක `sigmoid`.
   - **Multi-class Classification (බල්ලා/පූසා/මීයා - පන්ති 10ක් නම්):** Neurons 10 යි. Activation එක `softmax`.
   - **Regression (මිලක්, උෂ්ණත්වයක් අනුමාන කිරීම):** Neuron 1 යි. Activation එකක් නෑ (Linear).

3. **Hidden Layers (සැඟවුණු ස්ථර - මෙතන තමයි කලාව තියෙන්නේ):**
   - **කීයක් දානවද?** හැමවෙලේම සරලව පටන් ගන්න. මුලින්ම Hidden Layer 1ක් හෝ 2ක් විතරක් දාලා බලන්න. Model එකට ඉගෙනගන්න අමාරුයි වගේ නම් (Underfitting), තව Layers දාන්න.
   - **Neurons කීයක් දානවද?** සාමාන්‍යයෙන් පරිගණකයේ Memory එකට ලේසි වෙන්න 2යේ ගුණාකාර පාවිච්චි කරනවා (16, 32, 64, 128, 256).
   - **Funnel හැඩය (පුනීලයක් වගේ):** ගොඩක් වෙලාවට අපි කරන්නේ මුල්ම Hidden Layer එකට Neurons ගොඩක් දාලා (උදා: 128), ඊළඟට දාන ඒවා ක්‍රමයෙන් අඩු කරගෙන යන එකයි (උදා: 128 -> 64 -> 32 -> Output). මේකෙන් අර විසිරිලා තියෙන දත්ත ටික ක්‍රමයෙන් පෙරලා පෙරලා අන්තිම උත්තරේට අරන් එනවා.
   - **අනතුරු ඇඟවීමක්:** ඕනාවට වඩා ලොකුවට Layers/Neurons දැම්මොත් Model එක දත්ත ටික කටපාඩම් කරනවා (Overfitting). එහෙම වුණොත් Neurons ගාණ අඩු කරන්න, නැත්නම් `Dropout` දාන්න වෙනවා.

### Deep Learning වල තියෙන ප්‍රධාන Layers වර්ග සහ ඒවා පාවිච්චි කරන්නේ කොයි වෙලාවටද?

Deep Learning ලෝකයේ ප්‍රධාන වශයෙන් පාවිච්චි වෙන Layers ජාති කිහිපයක් තියෙනවා. අපේ දත්ත වල හැඩය (Data type) අනුව තමයි අපි මොන Layer එකද ගන්නේ කියලා තීරණය කරන්නේ:

1. **Dense Layer (Fully Connected Layer):**
   - **භාවිතා කරන්නේ:** සාමාන්‍ය වගුගත දත්ත වලට (Tabular data - CSV files, Excel sheets වල තියෙන දත්ත වගේ). ඒ වගේම ඕනෑම Network එකක (CNN, RNN වල පවා) අවසාන තීරණය (Classification) ගන්න පාවිච්චි කරන්නෙත් අනිවාර්යයෙන්ම මේකමයි.
   - **විශේෂත්වය:** මේකේ තියෙන හැම Neuron එකක්ම කලින් Layer එකේ හැම Neuron එකකටම වයර් වලින් සම්බන්ධ වෙලා තියෙනවා.

2. **Conv2D Layer (Convolutional Layer):**
   - **භාවිතා කරන්නේ:** පින්තූර (Images) සහ වීඩියෝ (Videos) වලටයි (Computer Vision).
   - **විශේෂත්වය:** මේකෙන් පින්තූරය තනි පේළියකට කඩන්නේ නෑ. ඒ වෙනුවට 2D Filter එකක් පාවිච්චි කරලා පින්තූරයේ තියෙන ලක්ෂණ (ඉරි, හැඩතල) උරාගන්නවා.

3. **RNN / LSTM / GRU Layers (Recurrent Layers):**
   - **භාවිතා කරන්නේ:** අනුපිළිවෙලක් (Sequence) තියෙන දත්ත වලටයි. ඒ කියන්නේ ඊයේ දත්තය අද දත්තයට බලපාන දේවල්. (උදා: කාලගුණ අනාවැකි කීම, Stock market අනුමාන කිරීම, භාෂා පරිවර්තනය - NLP, හඬ හඳුනාගැනීම - Audio).
   - **විශේෂත්වය:** සාමාන්‍ය Neural Network එකකට අතීතය මතක නෑ (No Memory). හැබැයි මේ Layers වලට තමන්ට කලින් ආපු වචනය මොකක්ද කියලා මතක තියාගන්න පුළුවන්.

4. **Dropout Layer:**
   - **භාවිතා කරන්නේ:** Model එක Overfit වෙන එක (දත්ත කටපාඩම් කරන එක) නවත්වන්නයි.
   - **විශේෂත්වය:** Training වෙලාවට අහඹු විදිහට (Randomly) සමහර Neurons ටිකක් නිදි කරවනවා (Turn off කරනවා). එතකොට අනිත් අයට කම්මැලි නැතුව තනියම ඉගෙනගන්න සිද්ධ වෙනවා.

5. **Flatten Layer:**
   - **භාවිතා කරන්නේ:** 2D හෝ 3D දත්ත (උදා: CNN එකකින් එන Output එක), 1D (තනි පේළියකට) හරවන්නයි.
   - **විශේෂත්වය:** මේකෙන් කිසිම දෙයක් ඉගෙනගන්නේ නෑ (Weights 0 යි). නිකම්ම හැඩය වෙනස් කරනවා විතරයි, මොකද Dense layer එකකට පින්තූරයක් ඒ විදිහටම දෙන්න බැරි නිසා.

6. **Embedding Layer:**
   - **භාවිතා කරන්නේ:** Natural Language Processing (NLP) වලදී, එහෙමත් නැත්නම් වචන (Text) එක්ක වැඩ කරද්දියි.
   - **විශේෂත්වය:** පරිගණකයට අකුරු තේරෙන්නේ නෑනේ. ඒ නිසා මේකෙන් කරන්නේ වචන අරගෙන, ඒ වචනයේ තේරුම ගැබ් වෙච්ච ඉලක්කම් ගොඩක් (Vector එකක්) විදිහට ඒක පරිවර්තනය කරන එකයි.

### ඇයි අපි හරියටම මේ තාක්ෂණික ක්‍රම (Techniques) තෝරගත්තේ?

අපි උඩ Code එකේදී නිකම්ම අහඹු විදිහට මේවා තෝරගත්තා නෙවෙයි. මේ එකින් එක තෝරගන්න ඉතාම පැහැදිලි, විද්‍යාත්මක හේතු තියෙනවා:

1. **Hidden Layer එකට `ReLU` ගත්තේ ඇයි?**
   - **හේතුව:** අද කාලේ Hidden Layers වලට පාවිච්චි කරන Standard (Default) Activation එක තමයි ReLU (Rectified Linear Unit).
   - **ගැඹුරු පැහැදිලි කිරීම:** අපි Sigmoid හරි Tanh හරි පාවිච්චි කළා නම් Vanishing Gradient (Gradient එක බිංදුව වෙලා ඉගෙනගන්න එක නවතින ප්‍රශ්නය) එන්න පුළුවන්. හැබැයි ReLU වලදී ධන අගයන්ට Gradient එක හැමවෙලේම 1යි. ඒ නිසා Network එක ගොඩක් වේගයෙන් Train වෙනවා. ඒ වගේම ගණනය කරන්න ගොඩක් ලේසියි (`max(0, x)` නිසා පරිගණකයට බරක් නෑ).

2. **Output Layer එකට `Sigmoid` ගත්තේ ඇයි?**
   - **හේතුව:** අපේ ප්‍රශ්නය Binary Classification (උත්තරේ 0 ද 1 ද කියලා හොයන එක). අපිට ඕනේ මේක 1 වෙන්න තියෙන සම්භාවිතාව (Probability) කීයද කියලා දැනගන්නයි.
   - **ගැඹුරු පැහැදිලි කිරීම:** Sigmoid එකෙන් කරන්නේ මොන තරම් ලොකු ධන අගයක් ආවත්, මොන තරම් කුඩා සෘණ අගයක් ආවත් ඒක හරියටම `0.0` ත් `1.0` ත් අතර පරාසයකට (Range) තද කරන එකයි. ඒ නිසා 0.85 ආවොත් අපිට කෙලින්ම කියන්න පුළුවන් 85% ක ෂුවර් එකක් තියෙනවා මේක Class 1 වෙන්න කියලා.

3. **Loss Function එකට `BinaryCrossentropy` ගත්තේ ඇයි (MSE නොගෙන)?**
   - **හේතුව:** Classification ප්‍රශ්න වලට (Sigmoid එකත් එක්ක) කවදාවත් Mean Squared Error (MSE) ගන්නේ නෑ. 
   - **ගැඹුරු පැහැදිලි කිරීම:** MSE පාවිච්චි කළොත් Loss ප්‍රස්ථාරයේ (Surface එකේ) වලවල් ගොඩක් හැදෙනවා (Non-convex වෙනවා). එතකොට Gradient Descent එක ඒ වලවල් වල හිරවෙනවා (Local Minima). 
   - හැබැයි Cross-Entropy වල තියෙන ලඝුගණක (Logarithms) නිසා, Model එක 100% ෂුවර් කියලා වැරදි උත්තරයක් දුන්නොත් අතිවිශාල දඬුවමක් (Penalty) දෙනවා. ඒ නිසා Loss Surface එක ලස්සන U හැඩයක් ගන්නවා, Gradient Descent එකට ලේසියෙන්ම පල්ලෙහාට යන්න පුළුවන් වෙනවා.

4. **Optimizer එක විදිහට `Adam` ගත්තේ ඇයි?**
   - **හේතුව:** Adam (Adaptive Moment Estimation) කියන්නේ Deep Learning වල රජා.
   - **ගැඹුරු පැහැදිලි කිරීම:** සාමාන්‍ය SGD පාවිච්චි කළා නම් ගොඩක් හෙමින් කන්දෙන් පල්ලෙහාට බහින්නේ (Oscillations වැඩියි). හැබැයි Adam කියන්නේ **Momentum** (බෝලයක් පෙරලෙනවා වගේ වේගය වැඩි කරන එක) සහ **RMSprop** (හැම Weight එකකටම වෙනම Learning Rate එකක් දෙන එක) කියන ක්‍රම දෙකේම එකතුවක්. ඒ නිසා ගොඩක් වෙලාවට Learning Rate එක වෙනස් කර කර වධ වෙන්න ඕනේ නෑ, Default අගයෙන්ම අනිත් හැම එකටම වඩා වේගයෙන් සහ නිවැරදිව වැඩ කරනවා.

---
## 3. The Forward Pass (ඉදිරි ගමන) සහ Loss ගණනය කිරීම

**Forward Pass** කියන්නේ Data ටික Model එක ඇතුලට දාලා, Weights වලින් ගුණ වෙලා, Activations හරහා ගිහින් අන්තිමට උත්තරයක් (Prediction එකක්) එළියට දෙන ක්‍රියාවලියටයි.

**Loss ගණනය කිරීම:** ඊටපස්සේ ඒ ආපු උත්තරේ (Predictions) සහ ඇත්ත උත්තරේ (y_train) අතර තියෙන වෙනස (Loss එක) අපි `loss_fn` එක පාවිච්චි කරලා ගණනය කරනවා.

In [4]:
# පලවෙනි දත්ත 5 අරගෙන Forward pass එකක් කරමු
X_sample = X_train[:5]
y_sample = y_train[:5]

# Forward Pass (Model එක අනුමාන කරන උත්තර)
predictions = model(X_sample)

# Cost (Loss) ගණනය කිරීම
cost = loss_fn(y_sample, predictions)

print("ඇත්ත උත්තර:\n", y_sample)
print("\nModel එකේ අනුමාන (Sigmoid හරහා):\n", predictions.numpy())
print("\nමේ දත්ත 5 සඳහා ආපු Cost (සාමාන්‍ය Loss එක):", cost.numpy())

# (තාම Train කරලා නැති නිසා Model එකේ උත්තර ගොඩක් දුරට වැරදියි)

ඇත්ත උත්තර:
 [[1.]
 [0.]
 [1.]
 [1.]
 [0.]]

Model එකේ අනුමාන (Sigmoid හරහා):
 [[0.48489678]
 [0.7954227 ]
 [0.6787759 ]
 [0.4902266 ]
 [0.7844985 ]]

මේ දත්ත 5 සඳහා ආපු Cost (සාමාන්‍ය Loss එක): 0.98915356


---
## 4. The Backward Pass (Backpropagation) - `tf.GradientTape` මගින්

දැන් අපි දන්නවා අපේ Model එක කොච්චර වැරදිද (Cost එක) කියලා. දැන් අපිට මේ වැරැද්ද අඩු කරන්න Weights වෙනස් කරන්න ඕනේ. ඒකට අපි Gradient Descent පාවිච්චි කරනවා.

Gradient එකක් හොයනවා කියන්නේ සරලවම ගණිතයේ එන **අවකලනය (Derivatives / Calculus)** කරනවා කියන එකයි (d(Loss) / d(Weight)). 
TensorFlow වලදී අපිට අතින් මේ අවකලනය කරන්න ඕනේ නෑ. ඒකට තියෙනවා **`tf.GradientTape()`** කියලා අපූරු මෙවලමක්. 

**Gradient Tape වැඩ කරන විදිහ:**
මේක හරියට Voice Recorder එකක් වගේ. අපි Tape එක "On" කරලා අර Forward Pass එකයි Loss එක ගණනය කරන එකයි කරනවා. එතකොට ඒක ඇතුලේ වෙන හැම ගුණ කිරීමක්ම, එකතු කිරීමක්ම Tape එකේ Record වෙනවා.
ඊටපස්සේ අපි Tape එකට කියනවා "මට අර Loss එකට අදාළව හැම Weight එකකම Gradient එක (අවකලනය) අරන් දෙන්න" කියලා. එයා Chain Rule එක පාවිච්චි කරලා ඒක Automatic කරලා දෙනවා!

In [5]:
# පියවර 1: Tape එක "On" කිරීම (Record කිරීම ආරම්භ කිරීම)
with tf.GradientTape() as tape:
    
    # පියවර 2: Forward Pass (Tape එක record කරනවා)
    predictions = model(X_sample)
    
    # පියවර 3: Loss ගණනය කිරීම
    loss = loss_fn(y_sample, predictions)

# පියවර 4: Record කරපු දේවල් පාවිච්චි කරලා Backpropagation (Gradients) ගණනය කිරීම
# model.trainable_variables කියන්නේ Model එකේ තියෙන ඔක්කොම Weights සහ Biases ටිකයි
gradients = tape.gradient(loss, model.trainable_variables)

print("Weights/Biases ගණන:", len(gradients))
print("\nපළවෙනි Layer එකේ Weights වල Gradients (අවකලන අගයන්):\n", gradients[0].numpy())

Weights/Biases ගණන: 4

පළවෙනි Layer එකේ Weights වල Gradients (අවකලන අගයන්):
 [[ 0.07528448 -0.16415782 -0.09055328 -0.00147101]
 [ 0.1008438  -0.5520456  -0.15112141 -0.08724376]]


---
## 5. Optimizer Step (Weights යාවත්කාලීන කිරීම)

දැන් Backpropagation හරහා අපිට Gradients ටික හම්බුණා. ඒ කියන්නේ "අපේ වැරැද්ද අඩු කරන්න නම් අහවල් Weight එක අහවල් පැත්තට අහවල් ප්‍රමාණයෙන් වෙනස් කරන්න ඕනේ" කියන උපදෙස් ටික අපිට හම්බෙලා තියෙන්නේ.

අපි දැන් ඒ උපදෙස් ටික (Gradients ටික) අපේ Optimizer එකට (අපි තෝරගත්ත Adam ට) දෙනවා. එයා අර අපි කලින් ඉගෙනගත්ත සමීකරණ (`New_Weight = Old_Weight - LR * Gradient`) පාවිච්චි කරලා Weights ටික Update කරනවා.

In [6]:
# zip මගින් gradients ටිකයි ඊට අදාළ weights ටිකයි ජෝඩු කරනවා (Pairs හදනවා)
optimizer.apply_gradients(zip(gradients, model.trainable_variables))

print("Optimizer එක මගින් Weights Update කළා! දැන් Model එක කලින්ට වඩා පොඩ්ඩක් හරි ඉගෙනගෙන තියෙනවා.")

Optimizer එක මගින් Weights Update කළා! දැන් Model එක කලින්ට වඩා පොඩ්ඩක් හරි ඉගෙනගෙන තියෙනවා.


---
## 6. The Complete Custom Training Loop (සම්පූර්ණ ක්‍රියාවලිය එකවර)

දැන් අපි ඉගෙනගත්ත දේවල් ඔක්කොම එකතු කරලා (Forward Pass -> Loss -> Tape/Backward Pass -> Optimizer Update) සම්පූර්ණ දත්ත ගොනුවටම Epochs 10ක් පුහුණු (Train) කරමු. 

මේක හරියටම Keras වල `model.fit()` ඇතුලේ වෙන දේම තමයි.

In [7]:
epochs = 10
batch_size = 32 # Mini-batch SGD පාවිච්චි කරන නිසා
dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train)).batch(batch_size)

for epoch in range(epochs):
    epoch_loss_avg = tf.keras.metrics.Mean()
    
    # සම්පූර්ණ Dataset එකම Batches වලට කඩලා එකින් එක ගන්නවා
    for x_batch, y_batch in dataset:
        
        # 1. Tape On
        with tf.GradientTape() as tape:
            # 2. Forward Pass
            predictions = model(x_batch, training=True) 
            # 3. Compute Loss
            loss = loss_fn(y_batch, predictions)
            
        # 4. Backward Pass (Get Gradients)
        grads = tape.gradient(loss, model.trainable_variables)
        
        # 5. Optimizer Update
        optimizer.apply_gradients(zip(grads, model.trainable_variables))
        
        # Loss එක record කරගන්නවා
        epoch_loss_avg.update_state(loss)
        
    print(f"Epoch {epoch+1:02d} | Cost (Average Loss): {epoch_loss_avg.result().numpy():.4f}")

Epoch 01 | Cost (Average Loss): 0.6976
Epoch 02 | Cost (Average Loss): 0.3918
Epoch 03 | Cost (Average Loss): 0.2372
Epoch 04 | Cost (Average Loss): 0.1659
Epoch 05 | Cost (Average Loss): 0.1301
Epoch 06 | Cost (Average Loss): 0.1089
Epoch 07 | Cost (Average Loss): 0.0948
Epoch 08 | Cost (Average Loss): 0.0846
Epoch 09 | Cost (Average Loss): 0.0769
Epoch 10 | Cost (Average Loss): 0.0708


---
## සාරාංශය (Summary)

1. **Forward Pass:** දත්ත Model එකට දීලා Weights සහ Activations හරහා ගිහින් පිළිතුරක් (Prediction) ගන්නවා.
2. **Loss:** අපේ Prediction එකයි ඇත්ත උත්තරෙයි වෙනස Loss Function එකෙන් මනිනවා.
3. **Backward Pass (GradientTape):** Loss එක අඩුවෙන්න නම් Weights වෙනස් වෙන්න ඕනේ කොහොමද කියන එක (Gradients / අවකලන) හොයාගන්නවා.
4. **Optimizer Update:** ඒ හොයාගත්ත Gradients අරගෙන Learning Rate එකට අනුව Weights ඇත්තටම වෙනස් කරනවා.

ඔයා `model.fit()` කෝල් කළාම, මේ Loop එක තමයි දහස් වාරයක් යටින් Run වෙන්නේ! අද ඉඳන් ඔයා හරියටම දන්නවා Deep Learning Model එකක් තිරයෙන් පිටුපස ඉගෙනගන්නේ කොහොමද කියලා.